# Exploring Ground Motion Model Space using Dimensionality Reduction

Ground Motion Models (GMM) make predictions of the median and standard deviation of ground motion for a large range of scenarios. As such they are high dimensional, with each GMM describing a unique set of values across the range of magnitudes, distances, intensity measures (IMs) and other source, path and site parameters required by the model. 

We can use visualisation tools such as trellis plots to explore how different GMMs compare with respect to certain predictors (e.g. magnitude, distance, period etc.) but given the high dimensionality it is more difficult to interpret how different from one another two or more models over the complete model space (i.e. range of scenarios). Understanding the extent to which models are different over the entire model space can be an important step in determining whether the choice of GMMs captures adequately centre, body and range of expected ground motions to capture the epistemic uncertainty in the hazard for a given site.

One tool that can help is _dimensionality reduction_, which describes a family of algorithms aimed at mapping a high dimensional data set into a low dimensional representation that preserves the relative model-to-model distances determined in the high dimensional space into the distances between objects in the low dimensional space. This allows us to visualise the relative proximity of models to one another as points in two- or three-dimensional space. One method for doing this is Sammon's mapping, which was introduced as a tool for GMM exploration and visualisation by Scherbaum et al. (2009) and has been used by several authors since (e.g. Goulet et al., 2021). However, other dimensionality reduction methods are available, some in the popular `scikit-learn` package for Python. So it can be interesting to see how the different dimensionality reduction methods compare.

We show here how to use the eGSIM API in a more advanced context to compare a larger set of GMMs developed within a single project/application (NGA East and the US National Seismic Hazard Model) and apply dimensionality reduction to see how the models are represented in low dimensional space.

<b>Pre-Requisites</b>
* We _strongly recommend_ that the user first familiarises themselves with the `Model-to-Model` notebook, which demonstrates the fundamentals of executing the eGSIM API for Model-to-Model Comparison
* The user will need to have the `scikit-learn` package installed within the environment where these notebooks are run. We recommend using a virtualenvironment (venv) for running the notebooks.

<b>Goals of the Notebook</b>

1) Use the eGSIM API to identify the NGAEast models from within the available list of models.

2) Use Model-to-Model function the eGSIM API to retrieve the expected ground motions from all the NGA East Models and compare them in a simple trellis plot.

3) Apply Sammon's Mapping to the expected ground motions from the NGA East Models and visualise them in a 2D plot.

4) Apply three other methods to the same dataset: Principal Component Analysis (PCA), t-Distributed Stochastic Neighbour Embedding (t-SNE) and Isomap  



<i>References</i>

```
Goulet, C. A., Bozorgnia, Y., Kuehn, N., Al Atik, L., Youngs, R. R., Graves, R. W., & Atkinson, G. M. (2021). NGA-East Ground-Motion Characterization model part I: Summary of products and model development. Earthquake Spectra, 37(1_suppl), 1231–1282. https://doi.org/10.1177/87552930211018723
```

```
Scherbaum, F., Kuehn, N. M., Ohrnberger, M., & Koehler, A. (2010). Exploring the Proximity of Ground-Motion Models Using High-Dimensional Visualization Techniques. Earthquake Spectra, 26(4), 1117–1138. https://doi.org/10.1193/1.3478697
```

## Setting up the Analysis

In [ ]:
%matplotlib inline

#Standard library imports for web API usage
import io
import requests
from typing import Optional

# Scientific python stack imports
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist  # Needed for Sammon's Mapping
import matplotlib.pyplot as plt

# Tools for dimensionality reduction (requires scikit-learn is installed)
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE, Isomap

In [ ]:
# Sammons mapping function taken from the implementation of https://github.com/tompollard/sammon
def sammon(
        x: np.ndarray,
        n: int,
        display: int = 2,
        inputdist: str = 'raw',
        maxhalves: int = 20,
        maxiter: int = 500, 
        tolfun: float = 1e-9,
        init: str = 'default'
):
    """Perform Sammon mapping on dataset x

    y = sammon(x) applies the Sammon nonlinear mapping procedure on
    multivariate data x, where each row represents a pattern and each column
    represents a feature.  On completion, y contains the corresponding
    co-ordinates of each point on the map.  By default, a two-dimensional
    map is created.  Note if x contains any duplicated rows, SAMMON will
    fail (ungracefully). 

    [y,E] = sammon(x) also returns the value of the cost function in E (i.e.
    the stress of the mapping).

    An N-dimensional output map is generated by y = sammon(x,n) .

    A set of optimisation options can be specified using optional
    arguments, y = sammon(x,n,[OPTS]):

       maxiter        - maximum number of iterations
       tolfun         - relative tolerance on objective function
       maxhalves      - maximum number of step halvings
       input          - {'raw','distance'} if set to 'distance', X is 
                        interpreted as a matrix of pairwise distances.
       display        - 0 to 2. 0 least verbose, 2 max verbose.
       init           - {'pca', 'cmdscale', random', 'default'}
                        default is 'pca' if input is 'raw', 
                        'msdcale' if input is 'distance'

    The default options are retrieved by calling sammon(x) with no
    parameters.

    File        : sammon.py
    Date        : 18 April 2014
    Authors     : Tom J. Pollard (tom.pollard.11@ucl.ac.uk)
                : Ported from MATLAB implementation by 
                  Gavin C. Cawley and Nicola L. C. Talbot

    Description : Simple python implementation of Sammon's non-linear
                  mapping algorithm [1].

    References  : [1] Sammon, John W. Jr., "A Nonlinear Mapping for Data
                  Structure Analysis", IEEE Transactions on Computers,
                  vol. C-18, no. 5, pp 401-409, May 1969.

    Copyright   : (c) Dr Gavin C. Cawley, November 2007.

    This program is free software; you can redistribute it and/or modify
    it under the terms of the GNU General Public License as published by
    the Free Software Foundation; either version 2 of the License, or
    (at your option) any later version.

    This program is distributed in the hope that it will be useful,
    but WITHOUT ANY WARRANTY; without even the implied warranty of
    MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the
    GNU General Public License for more details.

    You should have received a copy of the GNU General Public License
    along with this program; if not, write to the Free Software
    Foundation, Inc., 59 Temple Place, Suite 330, Boston, MA 02111-1307 USA

    """

    # Create distance matrix unless given by parameters
    if inputdist == 'distance':
        D = x
        if init == 'default':
            init = 'cmdscale'
    else:
        D = cdist(x, x)
        if init == 'default':
            init = 'pca'

    if inputdist == 'distance' and init == 'pca':
        raise ValueError("Cannot use init == 'pca' when inputdist == 'distance'")

    if np.count_nonzero(np.diagonal(D)) > 0:
        raise ValueError("The diagonal of the dissimilarity matrix must be zero")

    # Remaining initialisation
    N = x.shape[0]
    scale = 0.5 / D.sum()
    D = D + np.eye(N)     

    if np.count_nonzero(D<=0) > 0:
        raise ValueError("Off-diagonal dissimilarities must be strictly positive")   

    Dinv = 1 / D
    if init == 'pca':
        [UU,DD,_] = np.linalg.svd(x)
        y = UU[:,:n]*DD[:n] 
    elif init == 'cmdscale':
        from cmdscale import cmdscale
        y,e = cmdscale(D)
        y = y[:,:n]
    else:
        y = np.random.normal(0.0,1.0,[N,n])
    one = np.ones([N,n])
    d = cdist(y,y) + np.eye(N)
    dinv = 1. / d
    delta = D-d 
    E = ((delta**2)*Dinv).sum() 

    # Get on with it
    for i in range(maxiter):

        # Compute gradient, Hessian and search direction (note it is actually
        # 1/4 of the gradient and Hessian, but the step size is just the ratio
        # of the gradient and the diagonal of the Hessian so it doesn't
        # matter).
        delta = dinv - Dinv
        deltaone = np.dot(delta,one)
        g = np.dot(delta,y) - (y * deltaone)
        dinv3 = dinv ** 3
        y2 = y ** 2
        H = np.dot(dinv3,y2) - deltaone - np.dot(2,y) * np.dot(dinv3,y) + y2 * np.dot(dinv3,one)
        s = -g.flatten(order='F') / np.abs(H.flatten(order='F'))
        y_old    = y

        # Use step-halving procedure to ensure progress is made
        for j in range(maxhalves):
            s_reshape = np.reshape(s, (-1,n),order='F')
            y = y_old + s_reshape
            d = cdist(y, y) + np.eye(N)
            dinv = 1 / d
            delta = D - d
            E_new = ((delta**2)*Dinv).sum()
            if E_new < E:
                break
            else:
                s = 0.5*s

        # Bomb out if too many halving steps are required
        if j == maxhalves-1:
            print('Warning: maxhalves exceeded. Sammon mapping may not converge...')

        # Evaluate termination criterion
        if abs((E - E_new) / E) < tolfun:
            if display:
                print('TolFun exceeded: Optimisation terminated')
            break

        # Report progress
        E = E_new
        if display > 1:
            print('epoch = %d : E = %12.10f'% (i+1, E * scale))

    if i == maxiter-1:
        print('Warning: maxiter exceeded. Sammon mapping may not have converged...')

    # Fiddle stress to match the original Sammon paper
    E = E * scale
    
    return [y, E]


# Function to query the eGSIM API for Model-to-Model comparison
# This is the same function found in the `Model-to-Model` Notebook
def get_egsim_predictions(
        model: list[str],
        imt: list[str],
        magnitudes: list[float],
        distances: list[float],
        rupture_params: Optional[dict] = None,
        site_params: Optional[dict] = None,
        data_format="hdf",
) -> pd.DataFrame:
    """Retrieve the ground motion predictions for the selected set of ground motion
    models and intensity measure types. Each prediction will be the result of a given
    model, imt, and scenario, which is a configurable set of Rupture parameters and
    Site parameters.

    Args:
    - model: ground motion model(s) (OpenQuake class names)
    - imt: intensity measure type(s) (e.g. PGA, PGV, SA(0.1) etc.)
    - magnitudes: list of magnitudes configuring each Rupture
    - distances: list of distances configuring each Site
    - rupture_params: dict of shared Rupture parameters (magnitude excluded)
    - site_params: dict of shared Site parameters (distance excluded)
    - data_format: the requested data format. "hdf" (the default, recommended) or "csv".
      HDF is more performant and support more data types, but it requires pytables
      (`pip install tables`)

    Returns:

    A [pandas DataFrame](https://pandas.pydata.org/docs/user_guide/dsintro.html#dataframe)
    
    Each row denotes a scenario (i.e., a combination of a given Rupture and Site)
    labelled by a unique integer id, incremental and starting from 0 (*), and each column
    denotes:

    - a computed prediction if the first chunk of the column name is an intensity
      measure type (e.g. "PGA median BindiEtAl2014Rjb"): in this case, the second chunk
      is the metric type (e.g. "median") and the third the predicting model
      ("BindiEtAl2014Rjb")
    
    - a scenario input property if the first chunk of the column name is the text "input"
      (e.g., "input distance_measure rrup"): in this case, the second
      chunk is the configuration data type ("distance_measure", "intensity_measure",
      "rupture_parameter", "site_parameter" or "uncategorized") and the third is the
      configuration data name ("rrup")
    """
    base_url = "https://egsim.gfz.de/api/query/predictions"
    # request parameters (concatenate with site_config and rupture_config):
    parameters = {}
    if site_params:
        parameters |= site_params
    if rupture_params:
        parameters |= rupture_params

    # add remaining parameters:
    parameters |= {
        'model': model,
        'imt': imt,
        'format': data_format,
        'mag': magnitudes,
        'dist': distances
    }
    # POST request to eGSIM
    response = requests.post(base_url, json=parameters)

    # check HTTPErrors (status_code >= 400):
    if response.status_code >= 400 and response.text:
        raise requests.exceptions.HTTPError(response.text)
    else:
        response.raise_for_status()  # raises if status >= 400 (with a default message)

    # `response.content` is the computed data, as in-memory file (bytes sequence)
    # in CSV or HDF format. Read it into a pandas.DataFrame:
    if parameters['format'] == 'hdf':
        # `pd.read_hdf` works for HDF files on disk. Workaround:
        with pd.HDFStore(
                "data.h5",  # apparently unused for in-memory data
                mode="r",
                driver="H5FD_CORE",  # create in-memory file
                driver_core_backing_store=0,  # for safety, just in case
                driver_core_image=response.content) as store:
            dframe = store[list(store.keys())[0]]
    else:
        # use `pd.read_csv` with a BytesIO (file-like object) as input:
        dframe = pd.read_csv(io.BytesIO(response.content), header=[0, 1, 2], index_col=0)

    return dframe

## Find all of the NGA East GMMs available to eGSIM

The first task is to find out which GMMs available are part of the NGA East model suite. Fortunately, these are identifiable in their OpenQuake class names. So we can use eGSIM to provide a list of models that contain the label `NGAEastUSGS` in their class name. This will give us the list of GMMs we will use in the subsequent step

In [ ]:
# Get all the GMMs with the term "NGAEastUSGS"
r = requests.get("https://egsim.gfz.de/api/query/models?name=NGAEastUSGS")
if r.status_code == 200:
    query_result = list(r.json())
else:
    print(r.status_code)
    print(r.reason)
query_result

## Setup the eGSIM API Query and Retrieve the Expected Ground Motions and Standard Deviations

Here we will consider just 4 intensity measures (PGA, Sa(0.2 s), Sa(1.0 s) and Sa(2.0 s)) but retrieve the ground motions for a range of magnitudes from $4.0 \leq M_W \leq 8$ and a distances (not evenly spaced) between $1 \leq R_{RUP} (km) \leq 250$. We assume a common rock condition: $V_{S30} = 760$ m/s

In [ ]:
imts = ["PGA", "SA(0.2)", "SA(1.0)", "SA(2.0)"]

# Magnitudes between 4.0 and 8.0 spaced every 2.5 M units
magnitudes = np.arange(4.0, 8.0, 0.25).tolist()

# Distances
distances = [1.0, 2.0, 5.0, 7.5, 10.0, 15.0, 20.0, 30.0, 40.0, 50.0,
             60.0, 80.0, 100.0, 125.0, 150.0, 175.0, 200.0, 225.0, 250.0]

# Other rupture parameters
rupture_params = {
    "aspect": 1.5,
    "dip": 90.0,
    "ztor": 2.0,
    "rake": 0.0}
# Other site parameters
site_params = {
    "vs30": 760.0,
    "z1pt0": 25.0,
    "z2pt5": 1.2,
    "vs30measured": True
}

In [ ]:
# Get the ground motion values
nga_east_predictions = get_egsim_predictions(
    model=query_result, 
    imt=imts, 
    magnitudes=magnitudes,
    distances=distances,
    rupture_params=rupture_params,
    site_params=site_params,
)
nga_east_predictions

Now each GMM can be considered as a single observation point in a 304 dimensional space. Let's try to visualise this in 2D.

# Visualise the GMMS for a given scenario

The full suite of GMMs for the NGA East comprise a set of "seed" models (GMMs developed by individual developers/groups of developers) and "Sammons" models (a set of 17 GMMs designed to sample fully the model space) - See Goulet et al. (2021) for full details.

Let's separate out the "Seed" and "Sammons" Models. "Seed" models are assigned a code indicating the authors and assumptions, while the Sammons models are simply enumerated from 1 - 17.

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(12,12), sharex=True, sharey=True)
# Choose a given magnitude and compare the models in terms of attenuation with distance for each IMT
mag = 6.5
for imt, ax in zip(imts, axs.flatten()):
    
    idx = nga_east_predictions["input rupture_parameter mag"] == mag
    # Select only the predictions for the selected magnitude
    rrup = nga_east_predictions["input distance_measure rrup"][idx].to_numpy()
    for col in nga_east_predictions.columns:
        if ("stddev" in col) or (col.startswith("input")):
            continue
        col_imt, _, gmm_label = col.split(" ")
        if col_imt != imt:
            continue
        yvals = np.exp(nga_east_predictions[col][idx].to_numpy())
        if gmm_label.startswith("NGAEastUSGSSammons"):
            # Plot the Sammon models in blue
            ax.loglog(rrup, yvals, "-", color="tab:blue", lw=2)
        else:
            # Plot the seed models in red
            ax.loglog(rrup, yvals, "--", color="tab:red", lw=2)
    ax.grid(which="both")
    ax.set_xlim(1.0, 1000.0)
    ax.set_title(f"{imt} Mw = {mag}", fontsize=20)
    ax.tick_params(labelsize=12)
for i in range(2):
    axs[i, 0].set_ylabel("Acceleration (g)", fontsize=18)
    axs[1, i].set_xlabel("Rupture Distance (km)", fontsize=18)
fig.tight_layout()

#plt.savefig("./example_ngaeast_attenuation_Mw6p5.jpg", format="jpg", dpi=300, bbox_inches="tight")

# Generate a Sammons' Map for the GMMs

Sammons Mapping applies an iterative procedure to minimise the "sammons stress" ($E$), a measure of the relative misfit in the distance distribution in the low dimensional space with respect to the distance distribution in the high dimensional space.


Here we will apply the sammons mapping individually to each intensity measure.

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(12,12), sharex=True, sharey=True)
for imt, ax in zip(imts, axs.flatten()):
    median_gms_columns = []
    gmm_labels = []
    for col in nga_east_predictions:
        if f"{imt} median" in col:
            median_gms_columns.append(col)
            gmm_labels.append(col.split(" ")[-1])
    median_gm = nga_east_predictions[median_gms_columns].to_numpy().T
    # Normalize the median ground motions
    scaler = StandardScaler().fit(median_gm, )
    median_gm_norm = scaler.transform(median_gm)
    # Execute the sammons' map
    sammons_xy, sammons_stress = sammon(median_gm_norm, n=2, display=0)
    for gmm_label, (x, y) in zip(gmm_labels, sammons_xy):
        if "Sammons" in gmm_label:
            marker = "o"
            color="tab:blue"
            label_text = gmm_label.replace("NGAEastUSGSSammons", "Sam")
        else:
            marker = "s"
            color="tab:red"
            label_text = gmm_label.replace("NGAEastUSGSSeed", "")
        ax.plot([x], [y], marker=marker, color=color)
        ax.text(x, y, label_text)
    ax.grid(which="both")
    ax.set_title(f"{imt}", fontsize=20)
fig.tight_layout()
#plt.savefig("./example_ngaeast_sammons_map.jpg", format="jpg", dpi=300, bbox_inches="tight")

# Look at other dimensionality reduction methods

### Principal Component Analysis

Principal component analysis works differently from Sammons mapping but yields a comparable outcome. PCA applies a linear transformation to project data onto axes that describe the greatest amount of the variation.

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(12,12), sharex=True, sharey=True)
for imt, ax in zip(imts, axs.flatten()):
    median_gms_columns = []
    gmm_labels = []
    for col in nga_east_predictions:
        if f"{imt} median" in col:
            median_gms_columns.append(col)
            gmm_labels.append(col.split(" ")[-1])
    median_gm = nga_east_predictions[median_gms_columns].to_numpy().T
    # Normalize the median ground motions
    scaler = StandardScaler().fit(median_gm, )
    median_gm_norm = scaler.transform(median_gm)
    pca = PCA(n_components=2, svd_solver="full")
    xy = pca.fit_transform(median_gm_norm)
    for gmm_label, (x, y) in zip(gmm_labels, xy):
        if "Sammons" in gmm_label:
            marker = "o"
            color="tab:blue"
            label_text = gmm_label.replace("NGAEastUSGSSammons", "Sam")
        else:
            marker = "s"
            color="tab:red"
            label_text = gmm_label.replace("NGAEastUSGSSeed", "")
        ax.plot([x], [y], marker=marker, color=color)
        ax.text(x, y, label_text)
    ax.grid(which="both")
    ax.set_title(f"{imt}", fontsize=20)
fig.tight_layout()

There are differences, but in this case the low dimensional representation of the models using Sammons mapping and PCA are quite similar.

## t-Distributed Stochastic Neighbour Embedding & Isomap

The last two methods fall into the category of "manifold learning", which ultimately aims to project the points into positions on a manifold such that the distance between points reflects the relative *dissimilarity* between the high-dimensional points.

More information and scientfic references for the two methods can be found at their respective wikipedia pages (or the scikit-learn documentation)

**t-SNE** (https://en.wikipedia.org/wiki/T-distributed_stochastic_neighbor_embedding)

**Isomap** (https://en.wikipedia.org/wiki/Isomap

One key difference between these methods and those encountered so far (Sammons mapping & PCA) is that these methods require additional hyperparameters that control the quantification of dissimilarity here.

For **t-SNE** the hyperparameters are `perplexity` and `early_exaggeration`

For **Isomap** the hyperparameter is `n_neighbours`.

Refer to the Scikit-Learn documentation for further explanation (https://scikit-learn.org/stable/modules/manifold.html)

##### t-SNE

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(12,12), sharex=True, sharey=True)
for imt, ax in zip(imts, axs.flatten()):
    median_gms_columns = []
    gmm_labels = []
    for col in nga_east_predictions:
        if f"{imt} median" in col:
            median_gms_columns.append(col)
            gmm_labels.append(col.split(" ")[-1])
    median_gm = nga_east_predictions[median_gms_columns].to_numpy().T
    # Normalize the median ground motions
    scaler = StandardScaler().fit(median_gm, )
    median_gm_norm = scaler.transform(median_gm)
    # Hyperparameters perplexity and early exaggeration can be altered -
    # this will change the shape of the outputs
    tsne = TSNE(n_components=2, perplexity=20.0, early_exaggeration=12)
    xy = tsne.fit_transform(median_gm_norm)
    for gmm_label, (x, y) in zip(gmm_labels, xy):
        if "Sammons" in gmm_label:
            marker = "o"
            color="tab:blue"
            label_text = gmm_label.replace("NGAEastUSGSSammons", "Sam")
        else:
            marker = "s"
            color="tab:red"
            label_text = gmm_label.replace("NGAEastUSGSSeed", "")
        ax.plot([x], [y], marker=marker, color=color)
        ax.text(x, y, label_text)
    ax.grid(which="both")
    ax.set_title(f"{imt}", fontsize=20)
fig.tight_layout()

##### Isomap

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(12,12), sharex=True, sharey=True)
for imt, ax in zip(imts, axs.flatten()):
    median_gms_columns = []
    gmm_labels = []
    for col in nga_east_predictions:
        if f"{imt} median" in col:
            median_gms_columns.append(col)
            gmm_labels.append(col.split(" ")[-1])
    median_gm = nga_east_predictions[median_gms_columns].to_numpy().T
    # Normalize the median ground motions
    scaler = StandardScaler().fit(median_gm, )
    median_gm_norm = scaler.transform(median_gm)
    # Hyperparameter n_neighbors can be varied
    iso = Isomap(n_components=2, n_neighbors=10)
    xy = iso.fit_transform(median_gm_norm)
    for gmm_label, (x, y) in zip(gmm_labels, xy):
        if "Sammons" in gmm_label:
            marker = "o"
            color="tab:blue"
            label_text = gmm_label.replace("NGAEastUSGSSammons", "Sam")
        else:
            marker = "s"
            color="tab:red"
            label_text = gmm_label.replace("NGAEastUSGSSeed", "")
        ax.plot([x], [y], marker=marker, color=color)
        ax.text(x, y, label_text)
    ax.grid(which="both")
    ax.set_title(f"{imt}", fontsize=20)
fig.tight_layout()